# Minerva Genomic Language Model

![Minerva](https://proto-bio.github.io/proto-assets/images/tool/minerva/hero.png)

This notebook demonstrates the five functions of the Minerva tool. In the first section, we use `run_minerva_embeddings` to extract a locus embedding and per-token logits. In the second section, we use `run_minerva_score` to compute masked pseudo-log-likelihood metrics that measure how natural a locus appears to the model. In the third section, we use `run_minerva_sample` to resample masked positions while keeping each position's protein or DNA identity. In the fourth section, we use `run_minerva_gradient` to compute gradients of the masked log-likelihood with respect to a relaxed sequence. In the fifth section, we use `run_minerva_interactions` to predict maps of base pairing, protein contacts, and repeats across the locus. Minerva was initialized from gLM2 and reads a genomic locus as a mix of protein-coding genes and intergenic DNA; its interaction heads read out the coevolutionary signals it has learned, supporting alignment-free searches for structured non-coding RNAs and other genomic elements.

A locus is written with uppercase amino acids for each gene, lowercase `acgt` for intergenic DNA, and a `+` or `-` strand marker before each element.

In [ ]:
from pathlib import Path

from proto_tools.tools.masked_models.minerva import (
    MinervaEmbeddingsConfig, MinervaEmbeddingsInput, run_minerva_embeddings,
    MinervaScoringConfig, MinervaScoringInput, run_minerva_score,
    MinervaSampleConfig, MinervaSampleInput, run_minerva_sample,
    MinervaGradientConfig, MinervaGradientInput, run_minerva_gradient,
    MinervaInteractionsConfig, MinervaInteractionsInput, run_minerva_interactions,
)
from proto_tools.tools.masked_models.mixed_data_models import one_hot_mixed_logits
from proto_tools.transforms.masking import MaskingStrategy
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools

CHECKPOINT = "gbrixi/minerva-mlm"
DEVICE = "cuda"

display_available_tools("minerva")

## Prepared example locus

The protein and DNA segments below are shortened from the mixed-locus example in the [gLM2 upstream README](https://github.com/TattaBio/gLM2#usage). This illustrates the input notation.

The smaller fragment is used for the more expensive one-mask-per-position score and gradient calculations.

In [ ]:
SEQUENCE = "+MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP"
SHORT_SEQUENCE = "+MALTKV+aatt-MLGIDN"
print(SEQUENCE)

## Embeddings and logits

The pooled vector averages all unpadded tokens, including strand markers. Optional raw logits have one row per token and 24 columns in `ACDEFGHIKLMNPQRSTVWYacgt` order.

In [ ]:
display_api_reference("minerva-embedding", "input", "run_minerva_embeddings")
display_api_reference("minerva-embedding", "config", "run_minerva_embeddings")
display_api_reference("minerva-embedding", "output", "run_minerva_embeddings")

In [ ]:
embeddings_result = run_minerva_embeddings(
    MinervaEmbeddingsInput(sequences=[SEQUENCE]),
    MinervaEmbeddingsConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, return_logits=True,
    ),
)
assert embeddings_result.success, embeddings_result.errors
embedding = embeddings_result.results[0]
print("Pooled dimension:", len(embedding.mean_embedding))
print("Token count:", len(embedding.logits))
print("Logit columns:", embedding.vocab)
print("First tokens:", list(enumerate(SEQUENCE[:8], start=1)))

## Masked pseudo-log-likelihood

Every canonical protein/DNA position is masked in turn and scored from its context. Markers and ambiguous protein context are excluded from the score targets. Higher mean log-likelihood and lower perplexity indicate greater model predictability; they do not establish function.

In [ ]:
display_api_reference("minerva-score", "input", "run_minerva_score")
display_api_reference("minerva-score", "config", "run_minerva_score")
display_api_reference("minerva-score", "output", "run_minerva_score")

In [ ]:
score_result = run_minerva_score(
    MinervaScoringInput(sequences=[SHORT_SEQUENCE]),
    MinervaScoringConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1, return_logits=True,
    ),
)
assert score_result.success, score_result.errors
score = score_result.scores[0]
print("Mean log-likelihood:", score["avg_log_likelihood"])
print("Perplexity:", score["perplexity"])
print("Scored token positions:", score.scored_positions)

## Sampling while preserving modality

The automatic strategy selects two canonical biological positions. `fixed_positions=[2]` protects the first amino acid because the initial strand marker occupies position 1. The model may sample the original token, so two selected sites need not mean two substitutions.

In [ ]:
display_api_reference("minerva-sample", "input", "run_minerva_sample")
display_api_reference("minerva-sample", "config", "run_minerva_sample")
display_api_reference("minerva-sample", "output", "run_minerva_sample")

In [ ]:
sample_result = run_minerva_sample(
    MinervaSampleInput(sequences=[SEQUENCE]),
    MinervaSampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        masking_strategy=MaskingStrategy(num_mutations=2, fixed_positions=[2]),
        temperature=1.0,
        seed=7,
    ),
)
assert sample_result.success, sample_result.errors
sample = sample_result.results[0]
assert len(sample.sequence) == len(SEQUENCE)
assert sample.sequence[1] == SEQUENCE[1]
print("Original:", SEQUENCE)
print("Sampled: ", sample.sequence)

### Explicit masks with a protein/DNA label per site

An underscore does not carry case, so each premasked position needs an explicit modality. The mapping uses 1-indexed token positions. This example fills one protein and one DNA site through four refinement rounds. Explicit masks bypass automatic position selection.

In [ ]:
masked_tokens = list(SEQUENCE)
protein_position = 2
dna_position = next(i for i, token in enumerate(masked_tokens, start=1) if token in "acgt")
masked_tokens[protein_position - 1] = "_"
masked_tokens[dna_position - 1] = "_"

premasked_result = run_minerva_sample(
    MinervaSampleInput(
        sequences=["".join(masked_tokens)],
        mask_modalities=[{protein_position: "protein", dna_position: "dna"}],
    ),
    MinervaSampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        sampling_method="iterative_refinement",
        num_steps=4,
        top_p=0.9,
        temperature_annealing=False,
        seed=7,
    ),
)
assert premasked_result.success, premasked_result.errors
completed = premasked_result.results[0]
assert completed.sequence[protein_position - 1] in "ACDEFGHIKLMNPQRSTVWY"
assert completed.sequence[dna_position - 1] in "acgt"
print(completed.sequence)

## Relaxed-sequence gradient

The template fixes modality and strand markers. `one_hot_mixed_logits()` creates one 24-column row per token, with zero rows for fixed context. The default input temperature applies softmax within the allowed alphabet; use `temperature=None` only for already normalized modality-restricted probabilities. The model weights remain frozen.

In [ ]:
display_api_reference("minerva-gradient", "input", "run_minerva_gradient")
display_api_reference("minerva-gradient", "config", "run_minerva_gradient")
display_api_reference("minerva-gradient", "output", "run_minerva_gradient")

In [ ]:
gradient_result = run_minerva_gradient(
    MinervaGradientInput(
        inputs={"sequence": SHORT_SEQUENCE, "logits": one_hot_mixed_logits(SHORT_SEQUENCE, sharpness=2.0)},
        temperature=1.0,
    ),
    MinervaGradientConfig(model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1),
)
assert gradient_result.success, gradient_result.errors
candidate = gradient_result.results[0]  # one result per candidate; vocab stays on the output
term = candidate.terms[0]
assert term.gradient is not None
print("Masked mean NLL:", term.term_value)
print("Gradient shape:", (len(term.gradient), len(gradient_result.vocab)))
for token, row in zip(SHORT_SEQUENCE, term.gradient, strict=True):
    if token in ("+", "-"):
        assert all(value == 0.0 for value in row)

## Interaction maps

Each selected head returns its raw `(L, L)` probability matrix, with labels for both axes. Markers are included and padding is removed. No symmetry, threshold, diagonal removal, or strand averaging is imposed. All three heads are requested here; select fewer to reduce output size. The default two-layer heads can be changed with `interaction_layers=6`.

These are model interaction signals, not an experimentally established contact map or an RNA secondary-structure call.

In [ ]:
display_api_reference("minerva-interactions", "input", "run_minerva_interactions")
display_api_reference("minerva-interactions", "config", "run_minerva_interactions")
display_api_reference("minerva-interactions", "output", "run_minerva_interactions")

In [ ]:
interactions_result = run_minerva_interactions(
    MinervaInteractionsInput(sequences=[SEQUENCE]),
    MinervaInteractionsConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        heads=["base_pairing", "protein", "repeat"],
        interaction_layers=2,
    ),
)
assert interactions_result.success, interactions_result.errors
for head, interaction_map in interactions_result.results[0].maps.items():
    length = len(interaction_map.axis_labels)
    assert len(interaction_map.values) == length
    assert all(len(row) == length for row in interaction_map.values)
    print(head, (length, length), interaction_map.axis_labels[:5])

### Visualizing a structured RNA

`plot()` renders a result's maps as one heatmap: each cell takes the color of its strongest channel, with a track along the edges marking genes, intergenic DNA, and strand markers. Short toy loci carry little structure, so this example scores a 77-nt bacterial tRNA taken from the [Minerva README](https://github.com/garykbrixi/minerva). A tRNA folds into a cloverleaf, so its four base-paired stems should appear as short antidiagonal streaks in the base-pairing map.

In [ ]:
TRNA = "+cgcggggtggagcagcctggtagctcgtcgggctcataacccgaagatcgtcggttcaaatccggcccccgcaacca"

trna_result = run_minerva_interactions(
    MinervaInteractionsInput(sequences=[TRNA]),
    MinervaInteractionsConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
assert trna_result.success, trna_result.errors
fig = trna_result.plot()

Pass `channel` to view one map with a colorbar, and `window` to zoom into 1-indexed positions.

In [ ]:
fig = trna_result.plot(channel="base_pairing")

## Export

Exports are written to `example_output/` next to this notebook. Embedding CSV contains pooled vectors; score, sample, and gradient JSON preserve their result metadata.

Interaction maps default to compressed NPZ. Matrix and string-token arrays can be read with `allow_pickle=False`; JSON is also supported. Dense maps grow quadratically with token count.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export(name="minerva_embeddings", export_path=output_dir, file_format="csv")
score_result.export(name="minerva_scores", export_path=output_dir, file_format="json")
sample_result.export(name="minerva_samples", export_path=output_dir, file_format="json")
gradient_result.export(name="minerva_gradient", export_path=output_dir, file_format="json")
interactions_result.export(name="minerva_interactions", export_path=output_dir)
interactions_result.export(name="minerva_interactions", export_path=output_dir, file_format="json")

import numpy as np

with np.load(output_dir / "minerva_interactions.npz", allow_pickle=False) as archive:
    print("NPZ entries:", archive.files)
    print("Protein map shape:", archive["0_protein"].shape)
    print("Axis labels:", archive["0_protein_axis_labels"][:5])
print("Written files:", sorted(path.name for path in output_dir.iterdir()))